# Quaternion INDI on a Toy Quadrotor — an S0 Prototype Tour

This notebook is the illustrative companion to the a pure-NumPy prototype of the Tal–Karaman (TCST 2021) quaternion **INDI**
(Incremental Nonlinear Dynamic Inversion) trajectory-tracking controller,
validated in a toy rigid-body quadrotor sim before any firmware work.

The library does the real work — this notebook imports it and tells the story:

1. **Differential flatness** — attitude, body rates, and angular accelerations
   derived analytically from the trajectory (feedforward through *snap*).
2. **Phase-matched filtering** — the one INDI implementation detail that
   makes or breaks the whole scheme.
3. **A stability lesson** — how a physically inconsistent rotor inertia made
   the yaw loop non-minimum-phase unstable, and the algebra that explains it.
4. **Closed-loop tracking** — feedforward vs feedback-only, calm vs wind.
5. **Disturbance rejection without an integrator** — INDI's party trick.

Conventions throughout: **NED** world frame (gravity $+g\,\mathbf{e}_3$),
**FRD** body frame, Hamilton scalar-first quaternions mapping body→world,
thrust along $-\mathbf{z}_b$.


In [ ]:
import dataclasses
import numpy as np
import matplotlib.pyplot as plt

# indi_harness comes from the anixpkgs pin in .shell.nix (no path hacks).
from indi_harness import quat
from indi_harness.params import QuadParams
from indi_harness.trajectory import Hover, Circle, Lemniscate
from indi_harness.filters import Butter2
from indi_harness.flatness import flat_reference
from indi_harness.runner import run_scenario
from indi_harness.evalmetrics import rmse_position

P = QuadParams()
print(f"hover speed: {P.hover_speed():.1f} rad/s, "
      f"hover thrust {P.m * P.g:.2f} N on {P.m} kg")


## 1. Differential flatness: the whole reference state from the trajectory

A quadrotor's flat outputs are $(x, y, z, \psi)$. Given a trajectory with
derivatives through **snap**, the reference attitude $q_{ref}$, body rate
$\omega_{ref}$ (from the jerk chain) and body angular acceleration
$\dot\omega_{ref}$ (from the snap chain) follow in closed form — no
numeric differentiation of noisy signals, which is exactly what the Layer-B
feedforward path needs.

The S0 unit test for this is brutal and simple: the closed forms must match
finite differences of the flat attitude map itself. Let's reproduce it.


In [ ]:
lem = Lemniscate(amplitude=2.0, period=6.0, alt=5.0)
h = 1e-5
ts = np.linspace(0.3, 5.7, 200)
w_cl, w_fd, dw_cl, dw_fd = [], [], [], []
for t in ts:
    r0 = flat_reference(lem, t, P.m, P.g)
    rp = flat_reference(lem, t + h, P.m, P.g)
    rm = flat_reference(lem, t - h, P.m, P.g)
    w_cl.append(r0.w)
    w_fd.append(quat.qlog(quat.qmul(quat.qconj(rm.q), rp.q)) / (2 * h))
    dw_cl.append(r0.dw)
    dw_fd.append((rp.w - rm.w) / (2 * h))
w_cl, w_fd = np.array(w_cl), np.array(w_fd)
dw_cl, dw_fd = np.array(dw_cl), np.array(dw_fd)
print("max |omega_closed - omega_FD(attitude)| :", np.abs(w_cl - w_fd).max())
print("max |domega_closed - domega_FD(omega)|  :", np.abs(dw_cl - dw_fd).max())

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(ts, w_cl[:, i], label="closed form")
    ax.plot(ts[::10], w_fd[::10, i], ".", label="FD of attitude")
    ax.set_ylabel(["$\omega_x$", "$\omega_y$", "$\omega_z$"][i] + " [rad/s]")
axes[0].legend(); axes[0].set_title("Body-rate feedforward vs finite differences (lemniscate)")
axes[-1].set_xlabel("t [s]")
plt.tight_layout(); plt.show()


Machine-precision agreement. Fun fact from execution: the $\omega_z$ closed
form originally planned was **wrong in both sign conventions** found in the
literature; the FD test above is what caught it, and the correct form
($\omega_z = (\omega_x\,\mathbf{z}_b\!\cdot\!\mathbf{x}_c +
\dot\psi\,\mathbf{y}_b\!\cdot\!\mathbf{y}_c)/
\lVert\mathbf{z}_b\times\mathbf{x}_c\rVert$) was re-derived from
$\omega_z = -\mathbf{x}_b\cdot\dot{\mathbf{y}}_b$. Moral: make the test
the arbiter, not the paper.

## 2. Phase-matched filtering

INDI never inverts a full model. Each loop computes an **increment** from the
difference between a *commanded* and a *measured, filtered* derivative — e.g.
$\Delta\tau = J(\dot\omega_{cmd} - \dot\omega_f)$ — and adds it to the
*filtered actuator state*. The subtraction only cancels correctly if both
signals carry **identical filter phase lag**. Mismatched cutoffs produce an
oscillation that no gain tuning fixes (the design doc's #1 warning).

The library enforces this structurally: each loop's filters are built from a
single `(cutoff, fs)` pair. The property being protected is that an LTI filter
commutes with differentiation:


In [ ]:
fs, f_sig = 500.0, 3.0
t = np.arange(0, 4.0, 1 / fs)
x = np.sin(2 * np.pi * f_sig * t)
dx = 2 * np.pi * f_sig * np.cos(2 * np.pi * f_sig * t)

def filt(sig, fc):
    f = Butter2(fc, fs, 1)
    return np.array([f.update(np.array([v]))[0] for v in sig])

matched = filt(dx, 15.0)
mismatched = filt(dx, 40.0)             # the "same" signal, wrong cutoff
d_of_filtered = np.gradient(filt(x, 15.0), 1 / fs)

half = len(t) // 2
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(t[half:half+250], d_of_filtered[half:half+250], label="d/dt of filtered x  (15 Hz)")
ax.plot(t[half:half+250], matched[half:half+250], "--", label="filtered dx/dt  (15 Hz) — matched")
ax.plot(t[half:half+250], mismatched[half:half+250], ":", label="filtered dx/dt  (40 Hz) — mismatched")
ax.set_xlabel("t [s]"); ax.legend(loc="upper right")
ax.set_title("Phase matching: the increment pair must share one filter")
plt.tight_layout(); plt.show()
print("matched residual   :", np.abs(matched[half:] - d_of_filtered[half:]).max())
print("mismatched residual:", np.abs(mismatched[half:] - d_of_filtered[half:]).max())


The mismatched pair leaves a phase-shifted residual — inside an INDI increment
that residual becomes a persistent, gain-independent oscillation source.

## 3. A stability lesson: rotor inertia vs yaw authority

Quadrotor yaw torque comes from propeller drag ($k_m \Omega^2$) — slow, weak.
But *changing* rotor speed also produces an instantaneous inertial reaction
torque $-I_r \sum d_i \dot\Omega_i$. Commanding a yaw torque $\Delta\tau_z$
through the mixer therefore produces an immediate *opposing* torque with
magnitude ratio

$$ \rho = \frac{I_r}{2\,\bar\Omega\, k_m\, \tau_m} $$

For $\rho > 1$ the yaw loop is non-minimum-phase with gain > 1: closing it
fast is impossible. The original S0 plan picked $I_r = 6\times10^{-6}$ —
physically inconsistent with the small fast props implied by $k_f$ — giving
$\rho \approx 1.3$, which surfaced as a wind-triggered yaw limit cycle with
heavy allocator saturation. The fix was not gains but *physics*:
$I_r = 5\times10^{-7}$ ($\rho \approx 0.1$, matching real quads).


In [ ]:
def reaction_ratio(p):
    return p.Ir / (2 * p.hover_speed() * p.km * p.tau_m)

bad = dataclasses.replace(P, Ir=6.0e-6, J=P.J.copy(), drag_D=P.drag_D.copy())
print(f"reaction ratio: shipped Ir={P.Ir:.1e} -> rho={reaction_ratio(P):.2f}   "
      f"original plan Ir={bad.Ir:.1e} -> rho={reaction_ratio(bad):.2f}")

wind = np.array([3.0, 0.0, 0.0])
hover = Hover(point=np.array([0.0, 0.0, -5.0]))
runs = {}
for tag, plant in (("Ir = 5e-7 (consistent)", P), ("Ir = 6e-6 (inconsistent)", bad)):
    log = run_scenario(plant, hover, t_end=8.0, wind=wind, drag_on=True,
                       p0=np.array([0.0, 0.0, -5.0]))
    runs[tag] = log
    err = np.linalg.norm(log["p"] - log["p_ref"], axis=1)
    print(f"{tag}: steady err={err[log['t'] > 6].max():.3f} m, "
          f"saturation events={int(log['sat'].sum())}")

fig, ax = plt.subplots(figsize=(8, 3.2))
for tag, log in runs.items():
    yaw = np.array([np.rad2deg(quat.qlog(q)[2]) for q in log["q"]])
    ax.plot(log["t"], yaw, label=tag)
ax.set_xlabel("t [s]"); ax.set_ylabel("yaw [deg]"); ax.legend()
ax.set_title("Hover in 3 m/s wind: yaw limit cycle from non-minimum-phase rotor inertia")
plt.tight_layout(); plt.show()


## 4. Closed-loop tracking: feedforward earns its keep

The full cascade (linear INDI outer loop → tilt-prioritized quaternion
attitude → INDI rate loop → $\Omega^2$ allocation) flying the aggressive
Gerono lemniscate, with and without the flatness feedforward, calm and in a
3 m/s crosswind. Note the wind columns: INDI barely notices — the drag lives
in the accelerometer measurement, not in a model, and there is **no integrator
anywhere**.


In [ ]:
results = {}
for ff_on in (True, False):
    for windy in (False, True):
        log = run_scenario(P, lem, t_end=12.0,
                           wind=wind if windy else None, drag_on=True, ff_on=ff_on)
        _, total = rmse_position(log["t"], log["p"], log["p_ref"], trim_s=2.0)
        results[(ff_on, windy)] = (total, log)
        print(f"ff={str(ff_on):5s} wind={str(windy):5s} RMSE = {total:.4f} m")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
_, log_ref = results[(True, False)]
ax.plot(log_ref["p_ref"][:, 1], log_ref["p_ref"][:, 0], "k--", label="reference")
ax.plot(log_ref["p"][:, 1], log_ref["p"][:, 0], label="feedforward on")
_, log_fb = results[(False, False)]
ax.plot(log_fb["p"][:, 1], log_fb["p"][:, 0], label="feedback only")
ax.set_xlabel("E [m]"); ax.set_ylabel("N [m]"); ax.axis("equal"); ax.legend()
ax.set_title("Lemniscate tracking (calm, drag on)")
plt.tight_layout(); plt.show()


## 5. Disturbance rejection without an integrator

A 4 N lateral force pulse (≈ 0.4 g shove) for one second at hover. A PID would
need integral action to remove the offset — and pay for it with windup on
release. INDI's incremental structure gives integral-like rejection for free:
the un-modeled force appears in the measured specific force and is cancelled
increment by increment.


In [ ]:
pulse = lambda t: (np.array([4.0, 0.0, 0.0]) if 3.0 <= t < 4.0 else np.zeros(3))
log = run_scenario(P, hover, t_end=8.0, drag_on=True, f_ext=pulse,
                   p0=np.array([0.0, 0.0, -5.0]))
err = np.linalg.norm(log["p"] - log["p_ref"], axis=1)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(log["t"], err)
ax.axvspan(3.0, 4.0, alpha=0.15, label="4 N lateral pulse")
ax.set_xlabel("t [s]"); ax.set_ylabel("position error [m]"); ax.legend()
ax.set_title("Disturbance pulse: bounded excursion, clean recovery, zero offset")
plt.tight_layout(); plt.show()
print(f"peak excursion {err.max():.3f} m; residual after recovery "
      f"{err[log['t'] > 6.5].max():.4f} m")


## Where this goes next

This prototype is the S0 gate of a staged plan
(`~/dev/drone/sources/docs/superpowers/plans/2026-07-17-indi-s0-offline-prototype.md`):

- **S1** — the trajectory/evaluator half of `indi-harness` gets nix-packaged
  into anixpkgs; baseline RMSE for the *stock* ArduPilot controller in SITL.
- **S2** — this controller runs offboard as a ROS 2 node against SITL.
- **S3** — the validated math is transcribed into an `AC_CustomControl_INDI`
  backend in the ArduPilot fork (the Python here deliberately avoids scipy so
  that port is a transcription, not a redesign).
- **S4/S5** — aero-fidelity upgrade, robustness sweeps, and a CI RMSE gate in
  the `dronesim` NixOS VM test.

Primary references: Tal & Karaman (TCST 2021); Smeur, Chu & de Croon (JGCD
2016); Smeur, de Croon & Chu (CEP 2018); Sun et al. (T-RO 2022); Brescianini
& D'Andrea (TCST 2020).
